In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

from itertools import combinations

from utils.predict_aquarium_setup import predict_aquarium_setup
from utils.explain_pair_result import explain_pair_result
from utils.evaluate_community_tank import evaluate_community_tank
from utils.recommend_habitat import recommend_habitat

In [ ]:
species_df = pd.read_excel(
    "data/aquarium_fish_ml_dataset.xlsx",
    sheet_name="Species_Info"
)

pair_df = pd.read_excel(
    "data/aquarium_fish_ml_dataset.xlsx",
    sheet_name="Pair_Compatibility_ML"
)


In [ ]:
features = [
    "Same Water Type",
    "Temperature Overlap Ratio",
    "pH Overlap Ratio",
    "Salinity Overlap Ratio",
    "Tank Requirement Similarity",
    "Adult Size Ratio",
    "Predation Size Risk",
    "Temperament Risk",
    "Diet/Predation Risk",
    "Water Level Compatibility",
    "Schooling Conflict",
    "Reef Safety Conflict",
    "Same Group Aggression Conflict"
]

target = "Compatibility Label Code"

X = pair_df[features]
y = pair_df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

compatibility_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

compatibility_model.fit(X_train, y_train)

predictions = compatibility_model.predict(X_test)

In [ ]:
# Classification report
print("\nClassification Report:")
print(classification_report(
    y_test,
    predictions,
    target_names=["Not Compatible", "Use Caution", "Compatible"]
))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, predictions)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Not Compatible", "Use Caution", "Compatible"]
)

disp.plot(cmap="Blues", values_format="d")

plt.title("Fish Compatibility Model Confusion Matrix")
plt.xlabel("Predicted Compatibility")
plt.ylabel("Actual Compatibility")
plt.show()

In [ ]:
# Feature importance
feature_importance = pd.Series(
    compatibility_model.feature_importances_,
    index=X.columns
).sort_values(ascending=True)

feature_importance.plot(kind="barh", figsize=(8, 6))

plt.title("Feature Importance for Fish Compatibility Model")
plt.xlabel("Importance Score")
plt.ylabel("Model Feature")
plt.show()

In [ ]:
# Distribution of Fish Pairs
plt.figure(figsize=(8, 5))

plt.hist(
    pair_df["Overall Compatibility Score 0-100"],
    bins=10,
    edgecolor="black"
)

plt.title("Distribution of Fish Pair Compatibility Scores")
plt.xlabel("Compatibility Score")
plt.ylabel("Number of Fish Pairs")

plt.show()

In [ ]:
detailed_results = predict_aquarium_setup(
    ["Oscar", "Neon Tetra", "Guppy"],
    species_df,
    pair_df
)

detailed_results

In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(
    y_test,
    predictions,
    alpha=0.6
)

plt.xlabel("Actual Compatibility Label")
plt.ylabel("Predicted Compatibility Label")

plt.title("Actual vs Predicted Compatibility")

plt.grid(True)

plt.show()